In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from sklearn.linear_model import SGDClassifier

# Ensure imports resolve to the shared root utils package, not Models/utils.py
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "Models" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

sys.modules.pop("utils", None)
print(f"Project root: {PROJECT_ROOT}")

from utils.data import build_training_dataframe, load_clean_citation_dataframe
from utils.training import build_classification_report, incremental_sgd_step, split_train_test

Project root: C:\Users\Tommaso\Documents\MEGAR2D2\Documenti\SUPSI\BACHELOR\3_anno_bech\primaverile\M-P6203E-DataProjects-Hackaton3_P1


In [2]:
# PROJECT_ROOT already set in first cell
CHECKPOINTS_PATH = "../data/checkpoints/checkpoints/01_cleaned_data/01_cleaned_data_part_{}.parquet"

In [3]:
first_run = True
model = SGDClassifier(random_state=42, n_jobs=-1)

In [4]:
predictions = []
accuracies = []

In [ ]:
for i in range(1, 50):

    df = load_clean_citation_dataframe(
        CHECKPOINTS_PATH,
        start=i,
        end=i + 1,
        remove_empty_rows=True,
    )

    df_training = build_training_dataframe(
        df,
        seed=42 + i,
        filter_years=True,
    )

    step = incremental_sgd_step(
        model=model,
        df_training=df_training,
        first_run=first_run,
        random_state=42 + i,
        test_size=0.1,
        max_features=256,
        include_authors=True,
        classes=(0, 1),
    )

    model = step["model"]
    first_run = step["first_run"]

    predictions.extend(step["y_pred"])
    accuracies.append(step["accuracy"])

    X_model = step["X_model"]
    y_model = step["y_model"]
    feature_extractor = step["feature_extractor"]
    artifacts = step["artifacts"]

    print(f"Iteration {i}: Accuracy = {accuracies[-1]:.4f}")

    del df, df_training, step

Iteration 1: Accuracy = 0.3758
Iteration 2: Accuracy = 0.3728
Iteration 3: Accuracy = 0.6269
Iteration 4: Accuracy = 0.4108
Iteration 5: Accuracy = 0.4049
Iteration 6: Accuracy = 0.4604
Iteration 7: Accuracy = 0.5876
Iteration 8: Accuracy = 0.6214
Iteration 9: Accuracy = 0.4354
Iteration 10: Accuracy = 0.6280
Iteration 11: Accuracy = 0.4814
Iteration 12: Accuracy = 0.4716
Iteration 13: Accuracy = 0.6218
Iteration 14: Accuracy = 0.3788
Iteration 15: Accuracy = 0.3775
Iteration 16: Accuracy = 0.4176
Iteration 17: Accuracy = 0.6224
Iteration 18: Accuracy = 0.5528
Iteration 19: Accuracy = 0.6262
Iteration 20: Accuracy = 0.5728
Iteration 21: Accuracy = 0.6034
Iteration 22: Accuracy = 0.3841
Iteration 23: Accuracy = 0.6268
Iteration 24: Accuracy = 0.6158
Iteration 25: Accuracy = 0.3761
Iteration 26: Accuracy = 0.6213
Iteration 27: Accuracy = 0.6027
Iteration 28: Accuracy = 0.6197
Iteration 29: Accuracy = 0.5320
Iteration 30: Accuracy = 0.6273
Iteration 31: Accuracy = 0.5466
Iteration 32: Acc

In [ ]:
df = load_clean_citation_dataframe(
    CHECKPOINTS_PATH,
    start=55,
    end=56,
    remove_empty_rows=True,
)

df_training = build_training_dataframe(
    df,
    seed=42 + i,
    filter_years=True,
)

step = incremental_sgd_step(
    model=model,
    df_training=df_training,
    first_run=False,
    random_state=42 + i,
    test_size=0.99,
    max_features=256,
    include_authors=True,
    classes=(0, 1),
)

y_pred = step["y_pred"]
y_test = step["y_test"]
predictions.extend(y_pred)
accuracies.append(step["accuracy"])

class_report = build_classification_report(y_test, y_pred, target_names=("Not SDG", "SDG"))
print(class_report)

ValueError: The test_size = 1 should be greater or equal to the number of classes = 2

In [ ]:
X_train, X_test, y_train, y_test = split_train_test(
    X_model,
    y_model,
    test_size=0.99,
    random_state=42 + i,
    stratify=True,
)

y_pred = model.predict(X_test)
predictions.extend(y_pred)
accuracies.append((y_test == y_pred).mean())

class_report = build_classification_report(y_test, y_pred, target_names=("Not SDG", "SDG"))
print(class_report)

              precision    recall  f1-score   support

     Not SDG       0.41      0.65      0.51      5367
         SDG       0.67      0.44      0.53      8772

    accuracy                           0.52     14139
   macro avg       0.54      0.54      0.52     14139
weighted avg       0.57      0.52      0.52     14139

